# Dual-Wavelength PDPT Calibration — A Simulation Study**무엇을 하는 노트북인가**레이저 공정 중 표면 온도를 실시간으로 측정하는 PDPT(Photodiode-based Planck Thermometry) 시스템에서,두 파장의 검출기 전압비로부터 온도를 역산할 때 발생하는 다변수 오차를 어떻게 다룰 것인지 검토한다.**한계를 먼저 밝힌다**- 이 노트북은 **실측 데이터를 포함하지 않는다.** 캡스톤 과정에서 레이저 장비 고장으로 확보된 실측이  불완전했고, 여기서 사용하는 데이터는 전부 물리식으로부터 생성한 **합성 데이터**다.- 따라서 아래의 어떤 수치도 실제 계측 시스템의 성능이 아니다. 검증된 결과가 아니라,  **보정 전략을 세우기 위한 시뮬레이션 탐색**으로 읽어야 한다.**이 노트북의 이전 버전에는 데이터 리키지가 있었다**이전 버전은 입력 피처인 전류값을 예측 대상인 온도로부터 보간해 생성했다.즉 피처가 타겟의 변형에 불과했고, 그 결과 R²이 1.0으로 출력되었다.물리 계측 데이터에서 잔차가 사실상 0이라는 것은 모델이 잘 맞았다는 뜻이 아니라구조적 결함의 신호이므로, 원인을 역추적해 리키지를 확인하고 생성 과정을 다시 설계했다.본 버전은 (1) 인과 방향을 바로잡고, (2) 되돌릴 수 없는 물리적 산포를 명시적으로 넣고,(3) 리키지 자동 점검 절차를 학습 전에 통과시키도록 구성했다. 4절에 점검 절차가이전 버전의 결함을 실제로 검출하는지 확인하는 셀을 남겨두었다.

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom sklearn.model_selection import train_test_splitfrom sklearn.linear_model import LinearRegressionfrom sklearn.metrics import mean_squared_error, r2_scorefrom xgboost import XGBRegressorRNG = np.random.default_rng(42)# 물리 상수H_PLANCK = 6.62607015e-34   # J*sC_LIGHT  = 2.99792458e8     # m/sK_B      = 1.380649e-23     # J/KC2       = H_PLANCK * C_LIGHT / K_B   # 제2복사상수 [m*K]# 캡스톤에서 선정한 측정 파장LAM1 = 1180e-9   # mLAM2 = 1200e-9   # mprint(f"C2 = {C2:.6e} m*K")

## 1. 물리 모델 — 두 파장 복사비와 온도의 관계Wien 근사에서 분광 복사휘도는$$L_\lambda \propto \lambda^{-5}\exp\!\left(-\frac{C_2}{\lambda T}\right),\qquad C_2=\frac{hc}{k_B}$$두 파장의 비를 취하면 비례상수와(회색체 가정 하) 방사율이 소거된다.$$R=\frac{L_{\lambda_1}}{L_{\lambda_2}}=\left(\frac{\lambda_2}{\lambda_1}\right)^{5}\exp\!\left[\frac{C_2}{T}\left(\frac{1}{\lambda_2}-\frac{1}{\lambda_1}\right)\right]$$온도에 대해 정리하면$$T=\frac{C_2\left(\dfrac{1}{\lambda_2}-\dfrac{1}{\lambda_1}\right)}{\ln R+5\ln\left(\dfrac{\lambda_1}{\lambda_2}\right)}$$**부호에 관한 메모.** 캡스톤 보고서 식 (9)에는 분자가 $(1/\lambda_1-1/\lambda_2)$로 적혀 있으나,그 형태로 계산하면 온도가 음수가 된다. 위 유도에 따라 $(1/\lambda_2-1/\lambda_1)$가 맞으며,이 노트북은 직접 유도한 형태를 사용한다.

In [ ]:
def planck_ratio(T, lam1=LAM1, lam2=LAM2):    """Wien 근사 하 두 파장 복사휘도 비 L(lam1)/L(lam2)."""    return (lam2 / lam1) ** 5 * np.exp(C2 * (1.0 / lam2 - 1.0 / lam1) / T)def invert_ratio(R, lam1=LAM1, lam2=LAM2):    """복사비 R로부터 온도를 역산(물리식 단독 보정)."""    num = C2 * (1.0 / lam2 - 1.0 / lam1)    den = np.log(R) + 5.0 * np.log(lam1 / lam2)    return num / den# 왕복 검증: 역산이 원래 온도를 복원하는지T_check = np.array([900.0, 1200.0, 1686.0])print("복사비      :", np.round(planck_ratio(T_check), 5))print("역산 온도[K]:", np.round(invert_ratio(planck_ratio(T_check)), 3))

### 1.1 파장 간격의 트레이드오프두 파장이 가까울수록 방사율 비가 1에 수렴해 방사율 상쇄에는 유리하지만,동시에 두 신호가 비슷해져 **복사비의 온도 민감도**가 떨어진다.아래에서 $\lambda_1=1180\,\mathrm{nm}$를 고정하고 $\lambda_2$를 넓혀가며,온도 1 K를 분해하기 위해 복사비를 얼마나 정밀하게 측정해야 하는지 계산한다.

In [ ]:
T0 = 1400.0lam2_grid = np.linspace(1190e-9, 1400e-9, 60)req_precision = []for lam2 in lam2_grid:    dR = planck_ratio(T0 + 0.5, LAM1, lam2) - planck_ratio(T0 - 0.5, LAM1, lam2)    R0 = planck_ratio(T0, LAM1, lam2)    req_precision.append(abs(dR / R0))   # 1K 분해에 필요한 상대 정밀도req_precision = np.array(req_precision)plt.figure(figsize=(7, 4))plt.plot((lam2_grid - LAM1) * 1e9, req_precision * 1e6)plt.axvline(20, ls="--", c="crimson", label="Capstone choice (20 nm gap)")plt.xlabel("Wavelength gap $\\lambda_2-\\lambda_1$ [nm]")plt.ylabel("Required ratio precision\nfor 1 K resolution [ppm]")plt.title(f"Required measurement precision vs. wavelength gap (T = {T0:.0f} K)")plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()i20 = np.argmin(np.abs((lam2_grid - LAM1) * 1e9 - 20))print(f"간격 20 nm: 1 K 분해에 복사비를 {req_precision[i20]*1e6:.0f} ppm 정밀도로 측정해야 함")

## 2. 데이터 생성 — 인과 방향과 되돌릴 수 없는 산포**이전 버전이 잘못된 지점.** 전류를 온도로부터 보간해 만들었다. 전류는 실험자가 인가하는독립 변수이고 온도는 그 결과이므로, 인과가 뒤집혀 있었다.**방향만 바꾸면 되는가? 아니다.** 전류에서 온도를 *결정적 함수*로 계산하면리키지는 그대로 남는다. 실제로는 같은 전류를 인가해도 도달 온도가 달라진다.접촉저항 편차, 시편 기하 편차, 산화 진행에 따른 열손실 변화, 대류 조건이매 시행마다 다르기 때문이다. 이 **관측 불가능한 산포**가 있어야전류로부터 온도를 완벽히 복원할 수 없게 되고, 그때 비로소 리키지가 사라진다.생성 구조:```[독립 변수]  인가 전류 I,  주변 온도 T_amb,  경과 시간 t      |      |  줄 발열 - 열손실  (+ 관측 불가 산포: 접촉저항/기하 편차, 공정 노이즈)      v[참 온도]   T_true                <- 예측 대상      |      |  Planck 복사비  ->  검출기 전압      |  (+ 검출기 응답 불일치, 산화 드리프트, 암전류, 이상치)      v[측정값]    V1, V2  ->  측정 복사비 R_meas   <- 모델 입력```

In [ ]:
N = 3000# ---------- 독립 변수 (실험자가 정하거나 환경이 결정) ----------current_A   = RNG.uniform(4.5, 8.0, N)      # 인가 전류 [A]ambient_K   = RNG.uniform(293.0, 313.0, N)  # 주변 온도 [K]elapsed_s   = RNG.uniform(0.0, 600.0, N)    # 경과 시간 [s]# ---------- 관측 불가능한 산포 (여기가 리키지를 막는 핵심) ----------contact_var = RNG.normal(0.0, 0.045, N)     # 접촉저항/시편 기하 편차process_eps = RNG.normal(0.0, 22.0, N)      # 잔여 공정 노이즈 [K]# ---------- 참 온도: 줄 발열 - 손실 ----------JOULE_GAIN  = 40.0oxid_loss   = 0.035 * elapsed_s             # 산화 진행에 따른 열손실 증가 [K]T_true = (ambient_K          + JOULE_GAIN * (1.0 + contact_var) * current_A ** 1.7          - oxid_loss          + process_eps)print(f"참 온도 범위: {T_true.min():.0f} ~ {T_true.max():.0f} K")print(f"피처와 무관한 순수 잔여 노이즈(process_eps)의 표준편차: 약 {np.std(process_eps):.0f} K")print("(참고: contact_var는 current_A에 곱해지므로 모델이 이 항의 일부는 간접적으로 흡수할 수 있음.")print(" 따라서 이는 하한이 아니라, 어떤 모델도 완전히 설명할 수 없는 노이즈 성분의 참고치임.)")

### 2.1 측정 신호 — 계측계가 스스로 만드는 오차복사비 방식은 회색체 가정 하에서 방사율을 소거하지만, 실제 계측계에는 다음이 남는다.1. **검출기 응답 불일치** — 전압비를 복사휘도비로 치환하는 것은 두 파장에서 검출기 응답이   동일한 비례관계를 가질 때만 성립한다. 이 전제가 어긋나면 개별 측정이 아니라   산출된 온도 **전체**가 한쪽으로 치우친 계통 오차를 갖는다.2. **산화에 따른 분광 방사율 불일치 드리프트** — 표면이 산화되면 두 파장의 방사율이   완전히 같지는 않게 되어, 시간에 따라 비가 서서히 틀어진다.3. **암전류/배경** — 주변 온도에 의존하며, 신호가 약할수록 비를 더 크게 왜곡한다.4. **이상치** — 기계적 진동, 미광 유입.

In [ ]:
R_true = planck_ratio(T_true)# (1) 검출기 응답 불일치: 두 채널 이득비가 1이 아님 (계통 오차원)GAIN_MISMATCH = 1.006# (2) 산화에 따른 분광 방사율 불일치가 시간에 따라 누적emis_drift = 1.0 - 3.0e-5 * elapsed_s# (3) 암전류/배경: 주변 온도 의존, 신호 세기에 반비례해 비를 왜곡signal_strength = (T_true - 800.0) / 900.0dark_bias = 1.0 + 0.004 * (ambient_K - 303.0) / 10.0 / np.clip(signal_strength, .25, None)# (4) 검출 노이즈 + 이상치det_noise = RNG.normal(1.0, 0.0016, N)outlier   = np.where(RNG.random(N) < 0.02, RNG.normal(1.0, 0.02, N), 1.0)R_meas = R_true * GAIN_MISMATCH * emis_drift * dark_bias * det_noise * outlierdf = pd.DataFrame({    "R_meas":      R_meas,        # 측정 복사비    "ambient_K":   ambient_K,     # 주변 온도    "current_A":   current_A,     # 인가 전류 (독립 변수)    "elapsed_s":   elapsed_s,     # 경과 시간    "T_true":      T_true,        # 예측 대상})df.head()

## 3. 학습 전 리키지 점검모델을 돌리기 전에 **각 피처가 타겟의 결정적 변형이 아닌지** 먼저 확인한다.피처 하나만으로 타겟이 거의 완전히 설명된다면, 그 피처는 정보가 아니라타겟이 새어 들어온 것일 가능성이 높다.여기서는 각 피처를 단조 구간으로 나누어(비선형 관계도 잡아내기 위해)그 피처 **단독**으로 타겟을 얼마나 설명하는지 $R^2$로 측정하고,임계값을 넘으면 경고를 띄운다.

In [ ]:
def leakage_audit(X: pd.DataFrame, y: np.ndarray, threshold: float = 0.98, bins: int = 60):    """각 피처 단독의 설명력을 측정해 타겟 리키지 의심 피처를 표시한다.    선형 상관만 보면 비선형 리키지를 놓치므로, 피처를 분위수 구간으로 나눈 뒤    구간 평균으로 타겟을 예측하는 비모수 방식으로 단독 설명력을 계산한다.    """    rows = []    for col in X.columns:        v = X[col].to_numpy()        edges = np.quantile(v, np.linspace(0, 1, bins + 1))        idx = np.clip(np.digitize(v, edges[1:-1]), 0, bins - 1)        pred = np.zeros_like(y, dtype=float)        for b in range(bins):            m = idx == b            pred[m] = y[m].mean() if m.any() else y.mean()        r2 = r2_score(y, pred)        rows.append({            "feature": col,            "solo_R2": round(r2, 4),            "pearson": round(float(np.corrcoef(v, y)[0, 1]), 4),            "flag": "LEAK?" if r2 > threshold else "ok",        })    out = pd.DataFrame(rows).sort_values("solo_R2", ascending=False).reset_index(drop=True)    if (out["flag"] == "LEAK?").any():        print("경고: 타겟 리키지가 의심되는 피처가 있습니다. 생성 과정을 역추적하세요.")    else:        print("통과: 단독으로 타겟을 결정하는 피처가 없습니다.")    return outFEATURES = ["R_meas", "ambient_K", "current_A", "elapsed_s"]leakage_audit(df[FEATURES], df["T_true"].to_numpy())

### 3.1 점검 절차가 이전 버전의 결함을 실제로 잡아내는가점검 코드가 통과했다는 것만으로는 그 코드가 유효하다고 말할 수 없다.알려진 결함을 넣었을 때 실제로 잡아내는지 확인한다.아래는 이전 버전의 생성 방식(전류를 온도로부터 보간)을 그대로 재현한 것이다.

In [ ]:
# 이전 버전 재현: 문헌 실측점을 온도축으로 보간해 전류를 만들어냄paper_T_K     = np.array([297, 2459, 2526, 2595, 2661, 2734, 2802, 2871, 2943, 3012])paper_current = np.array([0, .3710, .3976, .4234, .4478, .4707, .4930, .5158, .5361, .5568])leaked_current = np.interp(T_true, paper_T_K, paper_current)   # <- 타겟에서 파생df_leaked = df.copy()df_leaked["current_A"] = leaked_currentprint("[이전 버전 생성 방식에 대한 점검 결과]")display(leakage_audit(df_leaked[FEATURES], df_leaked["T_true"].to_numpy()))

## 4. 보정 성능 비교 — 물리식 단독 역산 vs 다변수 보정**비교 조건을 명시한다.** 이전 버전은 피처 1개를 쓴 선형회귀와 피처 4개를 쓴 XGBoost를나란히 놓고 "17배 개선"이라고 적었다. 그 차이의 대부분은 알고리즘이 아니라**피처 개수**에서 온 것이므로 공정한 비교가 아니었다.여기서는 세 가지를 나누어 본다.| 방법 | 입력 | 무엇을 보는가 ||---|---|---|| A. 물리식 역산 | 측정 복사비만 | 보정 없이 이론식만 적용했을 때의 오차 || B. 선형 다변수 | 4개 피처 | 다변수 정보가 주는 이득 (알고리즘 효과 제외) || C. XGBoost | 4개 피처 | B와 동일 입력에서 비선형 모델의 추가 이득 |B와 C를 같은 입력으로 두어야 알고리즘의 기여를 분리해 볼 수 있다.

In [ ]:
X = df[FEATURES]y = df["T_true"].to_numpy()X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)def report(name, y_true, y_pred):    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))    return {"method": name, "RMSE_K": round(rmse, 2), "R2": round(r2_score(y_true, y_pred), 4)}results = []# A. 물리식 단독 역산 (학습 없음)pred_phys = invert_ratio(X_te["R_meas"].to_numpy())results.append(report("A. 물리식 역산 (복사비만)", y_te, pred_phys))# B. 선형 다변수lin = LinearRegression().fit(X_tr, y_tr)results.append(report("B. 선형 다변수 (4 피처)", y_te, lin.predict(X_te)))# C. XGBoost, 동일 입력xgb = XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.05,                   subsample=0.8, colsample_bytree=0.8, random_state=42)xgb.fit(X_tr, y_tr)results.append(report("C. XGBoost (4 피처)", y_te, xgb.predict(X_te)))res = pd.DataFrame(results)display(res)print(f"\n피처와 무관한 순수 잔여 노이즈(process_eps) 참고치: {np.std(process_eps):.0f} K "      "(엄밀한 하한은 아님, 2절 설명 참고)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))ax[0].scatter(y_te, pred_phys, s=8, alpha=.35, label="A. Physics-only inversion")ax[0].scatter(y_te, xgb.predict(X_te), s=8, alpha=.35, label="C. XGBoost")lims = [y_te.min(), y_te.max()]ax[0].plot(lims, lims, "r--", lw=1)ax[0].set_xlabel("True temperature [K]"); ax[0].set_ylabel("Predicted temperature [K]")ax[0].set_title("Actual vs. predicted"); ax[0].legend(); ax[0].grid(alpha=.3)ax[1].hist(pred_phys - y_te, bins=50, alpha=.6, label="A. Physics-only inversion")ax[1].hist(xgb.predict(X_te) - y_te, bins=50, alpha=.6, label="C. XGBoost")ax[1].axvline(0, c="r", ls="--", lw=1)ax[1].set_xlabel("Residual [K]"); ax[1].set_ylabel("Count")ax[1].set_title("Residual distribution -- physics-only bias is visible")ax[1].legend(); ax[1].grid(alpha=.3)plt.tight_layout(); plt.show()print(f"물리식 역산 잔차 평균: {np.mean(pred_phys - y_te):+.1f} K  (계통 오차)")print(f"XGBoost  잔차 평균: {np.mean(xgb.predict(X_te) - y_te):+.1f} K")

## 5. 결과 해석과 한계**관찰된 것**- 물리식 단독 역산의 잔차는 0을 중심으로 흩어지지 않고 한쪽으로 치우친다.  검출기 응답 불일치를 이득비 1.006으로 넣었기 때문이며, 이는 무작위 오차가 아니라  **계통 오차**다. 측정을 반복해도 줄어들지 않는다.- 주변 온도, 인가 전류, 경과 시간을 함께 입력하면 이 치우침의 상당 부분이 보정된다.  다만 개선의 대부분은 **다변수 정보** 자체에서 오며, 선형(B)과 XGBoost(C)의 차이는  그보다 작다. 알고리즘을 바꾸는 것보다 무엇을 함께 재느냐가 더 크게 작용했다.- 어떤 방법의 RMSE도 0에 가까워지지 않는다. 접촉저항·기하 편차, 잔여 공정 노이즈처럼  주어진 피처만으로는 설명되지 않는 산포를 데이터 생성 단계에서 의도적으로 남겨두었기  때문이다. XGBoost가 R²=1.0에 근접하지 않고 22K 수준의 잔차를 남긴 것 자체가,  3절의 자동 점검과는 별개로, 리키지가 없다는 정황적 신호다.**한계 — 이 노트북이 말하지 않는 것**1. 실측 데이터가 없다. 위 수치는 합성 데이터에 대한 것이며, 실제 시스템의 성능이 아니다.2. 오차 모델(이득 불일치 1.006, 드리프트 계수 등)은 실측으로 동정한 값이 아니라 가정값이다.   실제 계측계에서 이 값들을 먼저 동정해야 결론이 의미를 갖는다.3. 회색체 가정과 Wien 근사를 사용했다. 저온·장파장 영역에서는 Wien 근사의 오차가 커진다.4. 따라서 이 결과는 **보정 전략을 어디에 둘 것인지 정하기 위한 사전 검토**이며,   보정 곡선의 확정은 실측 확보 이후로 미룬다.**이 작업에서 남긴 것**성능 지표가 좋게 나왔을 때 그것을 근거로 삼기 전에, 그 숫자가 만들어진 경로를 먼저 확인해야 한다.이전 버전의 $R^2=1.0$은 모델이 잘 맞았다는 뜻이 아니라 피처가 타겟에서 파생됐다는 신호였다.3절의 점검 절차를 학습 파이프라인 앞에 두어, 같은 실수가 조용히 지나가지 않도록 했다.